In [1]:
import os
def read_prompt(name):
    with open(os.path.join('prompts', name), 'r') as f:
        prompt = f.read()
    return prompt 

In [2]:
# Создание модели
import os
from models import *
import json

def create_model(max_tokens=None, model_name = 'deepseek-chat'):
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not max_tokens:
        return DeepSeekSession(api_key, '', 
                               'You are an expert in resource constrained project scheduling problem algorithms. ' \
                               'Write and run code to answer questions if needed.', model_name=model_name)
    return DeepSeekSession(api_key, '', 
                           '''
You are an expert in resource constrained project scheduling problem algorithms.' \
Write clean, executable, production-ready Python code. \
                           ''',
                            max_tokens=max_tokens,
                            model_name=model_name)

def write_heuristics(dict_data, experiment = None):
    # Для словаря эвристик
    folder_path = os.path.join('Heuristics', experiment)
    os.makedirs(folder_path, exist_ok=True)
    for name, code in dict_data.items():
        with open(os.path.join(folder_path, name), "w", encoding="utf-8") as file:
            file.write(code)

def write_heuristic(code, name, experiment = None):
    # Для эвристики
    folder_path = os.path.join('Heuristics', experiment)
    os.makedirs(folder_path, exist_ok=True)
    with open(os.path.join(folder_path, name), "w", encoding="utf-8") as file:
        file.write(code)
    print(f'{name} save to {folder_path}')

def write_steps(dict_data, experiment = None):
    # Для шагов 
    folder_path = os.path.join('Heuristics', experiment, 'Steps')
    os.makedirs(folder_path, exist_ok=True)
    with open(os.path.join(folder_path, 'description.json'), 'w', encoding='utf-8') as f:
        json.dump(dict_data, f, ensure_ascii=False, indent=4)




In [4]:
import json
import re

def extract_code(code_text): 
    # Extract code from code block
    pattern = r'```(?:python)?\s*\n(.*?)```'
    code_blocks = re.findall(pattern, code_text, re.DOTALL)
    if code_blocks: 
        return code_blocks[0].strip() 
    else: 
        return None

### Method
def SGE(model_name,
        problem = 'RCPSP',
        EXPERIMENT = None):
    '''
    args:
    model (obj): Интерфейс к LLM-модели.
    problem (str): Тип RCPSP проблемы, RCPSP : default - классическая проблема.
    '''
    model = create_model(max_tokens=None, model_name = model_name)
    z_exp = read_prompt('1. exploration')
    z_decomposition = read_prompt('2. decomposition')
    z_integrate = read_prompt('3. integrate')

    folder_path = os.path.join('Heuristics', EXPERIMENT)
    os.makedirs(folder_path, exist_ok=True)
    
    #  Step 1 - Generate heuristics name
    prompt1 = z_exp.format(problem = problem)
    methods_text = model.send_message(prompt1)
    try:
        methods = json.loads(methods_text)
        assert isinstance(methods, list) and all(isinstance(m, str) for m in methods)
    except Exception as  e:
        print('Error parsing methods: {e}')
        methods = ['Tabu Search', 'Simulated Annealing']
    Q_N = methods[:10]
    print("Methods:", Q_N)
    # # Step 2 - Decomposition 
    Q_S = dict.fromkeys(Q_N)
    steps_raw = dict.fromkeys(Q_N)
    for k in range(len(Q_N)):
        Q_n = Q_N[k]
        prompt2 = z_decomposition.format(method=Q_n, problem = problem)
        steps_text = model.send_message(prompt2)
        steps_raw[Q_n] = steps_text
        try:
            steps = json.loads(steps_text)            
            assert isinstance(steps, list) and all(isinstance(s, str) for s in steps)
            steps_str = '\n'.join(steps)
            Q_S[Q_n] = steps_str 
        except Exception as e:
            print(f"Error parsing steps for method {Q_n}: {e}")
            continue
    print("Steps:", Q_S)
    write_steps(Q_S, EXPERIMENT)

    #
    # with open('Heuristics/deepseek_chat_free/Steps/description.json', "r", encoding="utf-8") as f:
    #     Q_S = json.load(f)

    #Q_S = {k: Q_S[k] for k in ['Minimum Slack Priority Rule (MSP)', 'Greatest Resource Demand Priority Rule (GRD)', 'Most Successors Priority Rule (MSP)']}
    #
    Q_A = dict.fromkeys(Q_S)
    # Step 3 - Write code 

    for method, steps_str in Q_S.items():
        coder = create_model(max_tokens=50_000, model_name = model_name)
        prompt3 = z_integrate.format(method=method, steps=steps_str)
        python_txt = extract_code(coder.send_message(prompt3))
        Q_A[method] = python_txt
        write_heuristic(python_txt, method, EXPERIMENT)
    # Code, Input to model, Steps
    return Q_A, Q_S
    # Step 3 - Generate Python-solution for interpreter
    pass

In [5]:
EXPERIMENT = 'deepseek_reasoner'

A,  S = SGE(model_name='deepseek-reasoner', EXPERIMENT=EXPERIMENT) 

# 18 m -> 12 alg
# 20 m -> 8 alg (reasoner)

Methods: ['Minimum Slack Priority Rule (MSP)', 'Greatest Resource Demand Priority Rule (GRD)', 'Most Successors Priority Rule (MSS)', 'Shortest Processing Time Priority Rule (SPT)', 'Longest Processing Time Priority Rule (LPT)', 'Earliest Start Time Priority Rule (EST)', 'Earliest Finish Time Priority Rule (EFT)', 'Latest Start Time Priority Rule (LST)', 'Latest Finish Time Priority Rule (LFT)', 'Greatest Rank Positional Weight Priority Rule (GRPW)']
Steps: {'Minimum Slack Priority Rule (MSP)': 'Step 1: Initialize parameters including job durations, resource requirements, resource capacities, and precedence relationships\nStep 2: Compute earliest start times (EST) and latest start times (LST) for all jobs using forward and backward passes without resource constraints\nStep 3: Calculate slack time for each job as Slack = LST - EST\nStep 4: Initialize current time t = 0, scheduled job list = empty, and active job list = empty\nStep 5: Identify eligible jobs (all predecessors completed an

# Valid heuristics
`Требования к эвристике:`

    - Временная сложность (timeout)
    - Ограничения по ресурсам (demand_min и т. д.)
    - Ограничения графа активностей (t_i > t_j, где t_i -- начало назначаемой работы должно быть позже времени окончания работы необходимых предшествующих работ j)

In [1]:
import tempfile

def my_interpter_solver(method, code, data):
    communication_coefficient = lambda n, m: 1.0 / (6 * m ** 2) * (-2 * n ** 3 + 3 * n ** 2 + (6 * m ** 2 - 1) * n) if m != 0 else 0.0
    with tempfile.NamedTemporaryFile(delete=False, suffix='.py') as temp_module:
        temp_module_name = os.path.basename(temp_module.name).split('.')[0]
        temp_module.write(code.encode('utf-8'))
        temp_module_path = temp_module.name
        jobs, resources_borders = data['jobs'], data['resources_detailed']
        try:
            # Import the module
            import importlib.util
            spec = importlib.util.spec_from_file_location(temp_module_name, temp_module_path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            # Check if 'rcpsp_solver' function exists
            if hasattr(module, 'rcpsp_solver'):
                schedule, order, resource_usage, d, makespan = module.rcpsp_solver(jobs, resources_borders, communication_coefficient)
                if not schedule or not order:
                    print(f"rcpsp_solver did not return a valid solution for method {method}")
            else:
                print(f"No rcpsp_solver function found in the code for method {method}")

            print('Solution:', makespan)

            return schedule, order, resource_usage, d, makespan

        except Exception as e:
                print(f"Error executing code for method {method}: {e}")
        finally:
                # Clean up the temporary file
                os.remove(temp_module_path)


In [3]:
import os
import json
from scripts.valid import *
import pandas as pd
from sampo.schemas.graph import WorkGraph
from sampo_api import *
from scripts.wg_converter import *




def test(heuristic, code, data):
    try:
        schedule, order, resource_usage, job_usage, makespan = my_interpter_solver(heuristic, code, data)
        return  (check_feasibility(schedule, job_usage, resource_usage,data), makespan)
    except Exception:
        return (False, -1)


def test1():
    # Проверка на простой кейс
    with open('testdata.json', 'r', encoding='utf-8') as file:
        data = json.load(file)['test0']
    return data
def test2():
    # Проверка на рабочий кейс
    wg = WorkGraph.loadf(f'wgs/small_synth', f'wg_9')
    contractors = contractor(N = 5)
    conv = WorkGraphConverter()
    data = conv.convert(wg, contractors)['rcpsp_data']
    return data
def test3():
    # Cycle for several problems
    pass


def test_heuristics(folder_path, heuristic):
    file = os.path.join(folder_path, heuristic)
    cases = dict.fromkeys(['name','test1', 'test2'])
    with open(file, "r", encoding="utf-8") as f:
        code = f.read()
    cases['name'] = heuristic
    # Test 1, assert makespan ~ 8
    data = test1()
    cases['test1'] = test(heuristic, code, data)
    # Test 2, wg_9.json, HEFT - 81, LFT - 58, GA - 59
    data = test2()
    cases['test2'] = test(heuristic, code, data)
    cases['test2_UPgap'] = round(100 * (cases['test2'][1] / 58 - 1))
    cases['test2_LOWgap'] = max( round(100 * (cases['test2'][1] / 81 - 1)), 0 )
    return cases
        
def test_prompt(EXPERIMENT):
    folder_path = os.path.join('Heuristics', EXPERIMENT)
    res = []
    for heuristic in os.listdir(folder_path):
        if heuristic != 'Steps':
            print('_' * 12, heuristic, '_' * 12)
           # check, makespan = test_heuristics(os.path.join(folder_path, heuristic))
            res.append( test_heuristics(folder_path, heuristic) )
    return res


df_test = pd.DataFrame(test_prompt('deepseek_reasoner'))

____________ Greatest Rank Positional Weight Priority Rule (GRPW) ____________
Solution: 6
Error executing code for method Greatest Rank Positional Weight Priority Rule (GRPW): Deadlock detected
____________ Longest Processing Time Priority Rule (LPT) ____________
Solution: 8
Solution: 136
____________ Latest Start Time Priority Rule (LST) ____________
Error executing code for method Latest Start Time Priority Rule (LST): Execution time limit exceeded
Error executing code for method Latest Start Time Priority Rule (LST): Execution time limit exceeded
____________ Minimum Slack Priority Rule (MSP) ____________
Solution: 8
Solution: 68
____________ Most Successors Priority Rule (MSS) ____________
Solution: 8
Solution: 79
____________ Shortest Processing Time Priority Rule (SPT) ____________
Solution: 8
Solution: 105
____________ Greatest Resource Demand Priority Rule (GRD) ____________
Solution: 8
Solution: 70
____________ Earliest Finish Time Priority Rule (EFT) ____________
Error execu

In [4]:
df_test

,name,test1,test2,test2_UPgap,test2_LOWgap
0,Greatest Rank Positional Weight Priority Rule ...,"(True, 6)","(False, -1)",-102,0
1,Longest Processing Time Priority Rule (LPT),"(True, 8)","(True, 136)",134,68
2,Latest Start Time Priority Rule (LST),"(False, -1)","(False, -1)",-102,0
3,Minimum Slack Priority Rule (MSP),"(True, 8)","(True, 68)",17,0
4,Most Successors Priority Rule (MSS),"(True, 8)","(True, 79)",36,0
5,Shortest Processing Time Priority Rule (SPT),"(True, 8)","(True, 105)",81,30
6,Greatest Resource Demand Priority Rule (GRD),"(True, 8)","(False, 70)",21,0
7,Earliest Finish Time Priority Rule (EFT),"(False, -1)","(False, -1)",-102,0
8,Earliest Start Time Priority Rule (EST),"(True, 8)","(True, 86)",48,6
9,Latest Finish Time Priority Rule (LFT),"(False, -1)","(False, -1)",-102,0


In [11]:
# with open('Heuristics/deepseek_chat/Earliest Start Time Priority Rule (EST)', "r", encoding="utf-8") as f:
#         code = f.read()
# data = test2()
# schedule, order, resource_usage, job_usage, makespan = my_interpter_solver('x', code, data)

# print( check_resource_feasibility(resource_usage, data['resources_detailed']) )

# print( find_capacity_violations(data['jobs'], data['resources_detailed'], schedule, job_usage) )



In [12]:
schedule['b96ae541-b974-a8fb-33df-a672d842b1d7']

NameError: name 'schedule' is not defined

In [ ]:
d['b96ae541-b974-a8fb-33df-a672d842b1d7']

In [ ]:
{d['id']:d for d in data['resources_detailed']}['cd613e30-d8f1-6adf-91b7-584a2265b1f5']

In [ ]:
schedule

In [ ]:
for j, (c, s, e) in schedule.items():
    if c != '025b413f-8a9a-021e-a648-a7dd06839eb9':
        continue
    print(j, c, s ,e )

# bd0b3ee0-dc22-cb5f-e710-486deaea75b4

# 5608d34c-5bb7-6d9a-e2ea-197c0d543207

In [ ]:
d['04a03bfe-884b-50ac-1cb3-0eb04d282944']

In [ ]:
d['bd0b3ee0-dc22-cb5f-e710-486deaea75b4']

In [ ]:
d['5608d34c-5bb7-6d9a-e2ea-197c0d543207']

In [ ]:
'a' in {'demand_max': {'a':1}}['demand_max']

In [ ]:
d

In [ ]:

df_test
# 1.3 - 2 Хуже от того что есть в начальной популяции

In [ ]:
# LFTScheduler().schedule(wg, contractors)[0].full_schedule_df.query('task_name == "KTP and NEP"')[['duration','workers']].values
# contractor(N=5)[0]
# wg = WorkGraph.loadf(f'wgs/small_synth', f'wg_9')
# wg.to_frame(save_req=True).query('activity_name == "KTP and NEP"')['req_volume'].values
# LFTScheduler().schedule(wg, contractors)[0].full_schedule_df

In [ ]:
# df_test

In [ ]:
# '''def create_prompt(rcpsp_data, 
#                   job_to_idx, 
#                   idx_to_job):
#     inp0 = 'You are given a list of jobs, where each row corresponds to an activity with precedence constraints and resource requirements.\n'
#     lines = ["row=|job_id|predecessors|priority|min_req|max_req|vol"]
#     for i, job in enumerate(rcpsp_data["jobs"]):
#         pred = [str(job_to_idx[j]) for j in job["predecessors"]]
#         if not pred:
#             pred = '[]'
#         else:
#             pred = '[' + ",".join(pred) + ']'
#         dmin = ",".join(f"{k}:{v}" for k, v in job["demand_min"].items())
#         dmax = ",".join(f"{k}:{v}" for k, v in job["demand_max"].items())
#         vol = ",".join(f"{k}:{v}" for k, v in job["work_volume"].items())
#         lines.append(f"|{i}|{pred}|{job['priority']}|{dmin}|{dmax}|{vol}") 
#     inp1 = "\n".join(lines)
#     inp2 = ';\nand a list contractors (tuple of workers) with maximum capacity:\n'
#     lines2 = []
#     for contractor in rcpsp_data["resources_detailed"]:
#          lines2.append(','.join(f"{r}:{v}" for r, v in contractor["workers"].items()))
#     inp3 = '\n'.join(lines2) + '. '
#     inp4 =  "The goal is to minimize the project makespan subject to precedence and renewable resource constraints."
#     return inp0 + inp1 + inp2 + inp3 + inp4

# print(create_prompt(data, job_to_idx, idx_to_job),)
# '''

# # # Converter for contractors
# # idx_contractors = range(1, len(contractors) + 1)
# # contractor_to_idx = {str(c.id):idx for c, idx in zip(contractors, idx_contractors)}
# # idx_to_contractor = {value: key for key, value in contractor_to_idx.items()}
# # # Converter for jobs
# # idx_to_job = wg.to_frame()['activity_id'].to_dict()
# # job_to_idx =  {value: key for key, value in idx_to_job.items()}